In [1]:
import numpy as np
import pandas as pd

### loading input data

In [ ]:
czetyl = pd.read_csv('czetyl.tsv', sep='\t')
paraczech_forms = pd.read_csv('../final-paraczech-package/paraczech/paraczech_forms.csv')
paraczech_lexes = pd.read_csv('../final-paraczech-package/paraczech/paraczech_lexemes_new.csv')

C:\Users\lukas\AppData\Local\Temp\ipykernel_36804\1879958477.py:3: DtypeWarning: Columns (3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  paraczech_lexes = pd.read_csv('../final-paraczech-package/paraczech/paraczech_lexemes_new.csv')


### recognition of borrowings

In [31]:
# labelling by tool
def origin_tool(x):
    if x == '#MISSING#':
        return 'loan'
    return 'native'

paraczech_forms['our_origin_tool'] = paraczech_forms['phon_form'].apply(origin_tool)

In [32]:
# labelling based on undeclensibility
def undeclensible(group):
    nom = group.loc[group['cell'] == 'nom.sg', 'orth_form']
    ins = group.loc[group['cell'] == 'ins.sg', 'orth_form']

    if nom.empty or ins.empty:
        return False
    
    return bool(set(nom) & set(ins))


undeclensible_lexemes = paraczech_forms.groupby('lexeme').filter(undeclensible)
paraczech_forms['our_origin_undeclensible'] = paraczech_forms['form_id'].isin(undeclensible_lexemes['form_id'])

In [ ]:
# labelling base on inflectional classes
loan_classes = paraczech_lexes[paraczech_lexes['pred_class'].isin(['N07', 'N16', 'N17', 'N18'])]
paraczech_forms['our_origin_classes'] = paraczech_forms['lexeme'].isin(loan_classes['label'])

In [ ]:
# (?) labelling based on derivational relatedness

### overlaps

In [54]:
df = paraczech_forms[['lexeme', 'our_origin_tool', 'our_origin_undeclensible', 'our_origin_classes']].drop_duplicates(subset=['lexeme']).copy()
df

,lexeme,our_origin_tool,our_origin_undeclensible,our_origin_classes
0,aachenský,native,False,False
72,aalborgský,native,False,False
113,aalen,loan,False,False
119,aalsmeerský,native,False,False
142,aarhuský,native,False,False
...,...,...,...,...
8318972,žvýkavost,native,False,False
8318974,žvýkavý,native,False,False
8319073,žvýknout,native,False,False
8319088,žvýknutí,native,False,False


In [72]:
print(df[(df['our_origin_tool'] == 'native') & (df['our_origin_undeclensible'] == True)].shape[0])
print(df[(df['our_origin_tool'] == 'native') & (df['our_origin_classes'] == True)].shape[0])

163
153


### finalisation of dataset with labelled borrowings

In [82]:
def finalisation(x):
    if x.our_origin_undeclensible:
        return 'loan'
    elif x.our_origin_classes:
        return 'loan'
    return x.our_origin_tool

df['our_origin_complex'] = df[['our_origin_tool', 'our_origin_undeclensible', 'our_origin_classes']].apply(finalisation, axis=1)

In [83]:
df

,lexeme,our_origin_tool,our_origin_undeclensible,our_origin_classes,our_origin_complex
0,aachenský,native,False,False,native
72,aalborgský,native,False,False,native
113,aalen,loan,False,False,loan
119,aalsmeerský,native,False,False,native
142,aarhuský,native,False,False,native
...,...,...,...,...,...
8318972,žvýkavost,native,False,False,native
8318974,žvýkavý,native,False,False,native
8319073,žvýknout,native,False,False,native
8319088,žvýknutí,native,False,False,native


### evaluation

In [84]:
df['czetyl_origin'] = df['lexeme'].map(czetyl.set_index('word')['is_loan'])

In [ ]:
df['czetyl_origin'].value_counts()

czetyl_origin
loan      5844
native    3297
Name: count, dtype: int64

In [89]:
def evaluation_metrics(column_to_evaluate):
    tp = df[(df[column_to_evaluate] == 'loan') & (df['czetyl_origin'] == 'loan')].shape[0]
    fp = df[(df[column_to_evaluate] == 'loan') & (df['czetyl_origin'] == 'native')].shape[0]
    fn = df[(df[column_to_evaluate] == 'native') & (df['czetyl_origin'] == 'loan')].shape[0]
    tn = df[(df[column_to_evaluate] == 'native') & (df['czetyl_origin'] == 'native')].shape[0]

    acc = (tp+tn)/(tp+tn+fp+fn)
    print(f'Accuracy={acc}')

    rec = tp/(tp+fn)
    print(f'Recall={rec}')

    pre = tp/(tp+fp)
    print(f'Precision={pre}')

    f1s = 2*((pre*rec)/(pre+rec))
    print(f'F1-score={f1s}')

In [93]:
print('TOOL:')
evaluation_metrics(column_to_evaluate='our_origin_tool')

print()

print('TOOL+MORPHOLOGY:')
evaluation_metrics(column_to_evaluate='our_origin_complex')

TOOL:
Accuracy=0.8199321737227875
Recall=0.7375085557837098
Precision=0.9746720940750792
F1-score=0.8396649133060589

TOOL+MORPHOLOGY:
Accuracy=0.8217919264850673
Recall=0.7417864476386037
Precision=0.9730639730639731
F1-score=0.8418293038159044


#### qualitative observations

In [97]:
df[(df['czetyl_origin'] == 'loan') & (df['our_origin_tool'] == 'native')]

,lexeme,our_origin_tool,our_origin_undeclensible,our_origin_classes,our_origin_complex,czetyl_origin
9568,achát,native,False,False,native,loan
10888,adamita,native,False,False,native,loan
33707,ajnclík,native,False,False,native,loan
39907,akr,native,False,False,native,loan
41613,aksamit,native,False,False,native,loan
...,...,...,...,...,...,...
8313647,žold,native,False,False,native,loan
8314547,žoržet,native,False,False,native,loan
8315616,žrec,native,False,False,native,loan
8315936,žula,native,False,False,native,loan


In [101]:
df[(df['czetyl_origin'] == 'loan') & (df['our_origin_complex'] == 'native')]

,lexeme,our_origin_tool,our_origin_undeclensible,our_origin_classes,our_origin_complex,czetyl_origin
9568,achát,native,False,False,native,loan
10888,adamita,native,False,False,native,loan
33707,ajnclík,native,False,False,native,loan
39907,akr,native,False,False,native,loan
41613,aksamit,native,False,False,native,loan
...,...,...,...,...,...,...
8313647,žold,native,False,False,native,loan
8314547,žoržet,native,False,False,native,loan
8315616,žrec,native,False,False,native,loan
8315936,žula,native,False,False,native,loan
